# 공통 유틸리티

이 노트북은 Stage 1에서 구현한 `src/utils/` 패키지의 5개 유틸리티(batching, random, io, checkpoints, training_plots)를 직접 실행하고 검증하는 실습 자료이다.
이전 노트북의 변수나 실행 결과를 사용하지 않으며, 이 노트북 단독으로 완전히 실행할 수 있다.

**실행 환경**: `conda run -n numpy_py311` (GPU 불필요)

**목표**
- `get_batches`로 이미지와 레이블 배열을 mini-batch로 분할하고 shuffle 동작을 확인한다.
- `set_seed`로 난수 시드를 고정하여 실험 재현성을 검증한다.
- `save_params`/`load_params`로 numpy 배열 dict를 `.npz`로 저장하고 복원한다.
- `checkpoints.save`/`load`로 모델 파라미터를 in-place 저장·복원한다.
- `plot_training_log`로 epoch별 loss/metric 곡선을 PNG로 저장한다.

## 0. 환경 설정

In [ ]:
# sys.path setup -- excluded from jupyter book build
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"project_root={PROJECT_ROOT}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.utils.batching import get_batches
from src.utils.random import set_seed
from src.utils.io import save_params, load_params
from src.utils import checkpoints
from src.utils.training_plots import plot_training_log

## 1. 개요

`src/utils/`는 Stage 2 이후 모든 Stage에서 공통으로 사용하는 보조 모듈을 제공한다.
학습 루프, 체크포인트 관리, 결과 시각화 등 구현 로직과 무관한 반복 작업을 이 패키지로 분리하여 각 Stage 코드를 단순하게 유지한다.

이 패키지가 제공하는 유틸리티는 다음과 같다.

| 모듈 | 핵심 API | 역할 |
|---|---|---|
| `batching.py` | `get_batches` | 배열을 mini-batch로 분할, shuffle 지원 |
| `random.py` | `set_seed` | numpy 난수 시드 고정 |
| `io.py` | `save_params`, `load_params` | dict를 .npz로 저장·복원 |
| `checkpoints.py` | `save`, `load` | 모델 params를 .npz로 저장·in-place 복원 |
| `training_plots.py` | `plot_training_log` | loss/metric 곡선을 PNG로 저장 |

## 2. mini-batch 분할

`get_batches`는 하나 이상의 numpy 배열을 받아 `batch_size` 크기의 mini-batch를 순서대로 yield하는 generator이다.
`shuffle=True`이면 모든 배열에 동일한 인덱스를 적용하여 이미지와 레이블의 쌍 관계를 유지하면서 순서를 섞는다.
배열이 1개이면 `np.ndarray`를 yield하고, 2개 이상이면 `tuple`을 yield한다.

In [ ]:
set_seed(42)

# synthetic dataset: 100 samples, 784 features, 10 classes
N, D = 100, 784
images = np.random.randn(N, D).astype(np.float32)
labels = np.arange(N) % 10  # labels 0~9 repeating

print(f"images.shape={images.shape}, labels.shape={labels.shape}")

# shuffle=False: original order
print("\nshuffle=False - first 3 batches (labels):")
for i, (x, y) in enumerate(get_batches(images, labels, batch_size=32)):
    print(f"  batch {i}: x.shape={x.shape}, y[:4]={y[:4]}")

# shuffle=True: random order
print("\nshuffle=True - first batch (labels):")
for x, y in get_batches(images, labels, batch_size=32, shuffle=True):
    print(f"  x.shape={x.shape}, y[:4]={y[:4]}")
    break

In [ ]:
# batch size 검증
batches = list(get_batches(images, labels, batch_size=32))
assert len(batches) == 4, f"expected 4 batches, got {len(batches)}"
assert all(len(x) <= 32 for x, y in batches), "batch size exceeded"
assert sum(len(x) for x, y in batches) == N, "total sample count mismatch"

# 마지막 배치 크기 검증 (100 = 32*3 + 4)
assert len(batches[-1][0]) == 4, f"expected last batch size=4, got {len(batches[-1][0])}"

# shuffle=True: 인덱스 동기화 검증
set_seed(0)
x_batch, y_batch = next(iter(get_batches(images, labels, batch_size=32, shuffle=True).__iter__() if False else
                              (b for b in get_batches(images, labels, batch_size=32, shuffle=True))))
# images[i]의 label이 i % 10 이므로, x_batch의 행과 y_batch 값이 일치해야 한다
for img, lbl in zip(x_batch[:5], y_batch[:5]):
    matched = any(np.allclose(img, images[j]) and labels[j] == lbl for j in range(N))
    assert matched, "image-label pair mismatch after shuffle"

print("get_batches validation passed")

## 3. 난수 시드 고정

`set_seed`는 `np.random.seed`를 호출하여 numpy 난수 생성기 상태를 초기화한다.
같은 시드로 호출하면 이후 생성되는 모든 난수 배열이 동일한 순서를 따르므로 실험 조건을 완전히 재현할 수 있다.

In [ ]:
# 같은 시드로 두 번 호출하면 동일한 배열 생성
set_seed(42)
a = np.random.randn(5)

set_seed(42)
b = np.random.randn(5)

print(f"a = {a}")
print(f"b = {b}")
print(f"a == b: {np.allclose(a, b)}")

# 다른 시드로 호출하면 다른 배열 생성
set_seed(99)
c = np.random.randn(5)
print(f"\nc (seed=99) = {c}")
print(f"a == c: {np.allclose(a, c)}")

In [ ]:
set_seed(42)
x1 = np.random.randn(10)

set_seed(42)
x2 = np.random.randn(10)

assert np.allclose(x1, x2), "same seed must produce identical arrays"

set_seed(0)
x3 = np.random.randn(10)
assert not np.allclose(x1, x3), "different seeds must produce different arrays"

print("set_seed validation passed")

## 4. 파일 입출력

`save_params`는 문자열 키와 numpy 배열 값으로 구성된 dict를 `.npz` 파일로 저장한다.
`load_params`는 `.npz` 파일을 읽어 동일한 구조의 dict로 반환한다.
`.npz` 형식은 여러 배열을 하나의 파일에 키 기반으로 저장하므로 모델 파라미터처럼 여러 배열을 묶어 관리할 때 적합하다.

In [ ]:
import tempfile
import pathlib

# save and load params dict
params = {
    "w": np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32),
    "b": np.array([0.5, -0.5], dtype=np.float32),
}

with tempfile.TemporaryDirectory() as tmpdir:
    path = str(pathlib.Path(tmpdir) / "params.npz")
    save_params(params, path)
    print(f"saved to: {path}")

    loaded = load_params(path)
    print(f"loaded keys: {list(loaded.keys())}")
    print(f"w:\n{loaded['w']}")
    print(f"b: {loaded['b']}")

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = str(pathlib.Path(tmpdir) / "test.npz")
    save_params(params, path)

    assert pathlib.Path(path).exists(), "file not created"

    loaded = load_params(path)
    assert isinstance(loaded, dict), "load_params must return dict"
    assert set(loaded.keys()) == set(params.keys()), "key mismatch"
    assert np.allclose(loaded["w"], params["w"]), "w value mismatch"
    assert np.allclose(loaded["b"], params["b"]), "b value mismatch"

print("save_params/load_params validation passed")

## 5. 모델 체크포인트

`checkpoints.save`는 `model.params` 리스트를 `param_0`, `param_1`, ... 키로 `.npz`에 저장한다.
`checkpoints.load`는 저장된 배열을 동일한 인덱스로 꺼내 `param[...] =` 연산으로 in-place 복원한다.
CuPy 배열은 저장 시 자동으로 NumPy로 변환되고, 복원 시 원래 배열 모듈로 되돌아간다.

여기서는 `model.params`와 동일한 구조(리스트 속 numpy 배열)를 가진 간단한 객체로 테스트한다.

In [ ]:
class SimpleModel:
    """Minimal model stub for checkpoint testing."""
    def __init__(self):
        self.params = [
            np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32),  # w0
            np.array([0.1, 0.2], dtype=np.float32),                # b0
            np.array([[5.0], [6.0]], dtype=np.float32),            # w1
            np.array([0.3], dtype=np.float32),                     # b1
        ]

model = SimpleModel()
print("original params:")
for i, p in enumerate(model.params):
    print(f"  param_{i}: shape={p.shape}, values={p.ravel()[:3]}")

with tempfile.TemporaryDirectory() as tmpdir:
    path = str(pathlib.Path(tmpdir) / "model")
    checkpoints.save(model, path)
    print(f"\ncheckpoint saved: {path}.npz")

    # modify params to simulate a new state
    for p in model.params:
        p[...] = 0.0
    print("\nafter zeroing params[0]:", model.params[0].ravel()[:3])

    # restore
    checkpoints.load(model, path)
    print("\nafter load - params[0]:", model.params[0].ravel()[:3])

In [ ]:
original_values = [p.copy() for p in model.params]

with tempfile.TemporaryDirectory() as tmpdir:
    path = str(pathlib.Path(tmpdir) / "ckpt")
    checkpoints.save(model, path)

    # verify file created with .npz extension
    assert pathlib.Path(path + ".npz").exists(), ".npz file not created"

    # overwrite params
    for p in model.params:
        p[...] = 99.0

    checkpoints.load(model, path)

    for i, (orig, restored) in enumerate(zip(original_values, model.params)):
        assert np.allclose(orig, restored), f"param_{i} restore mismatch"

print("checkpoints.save/load validation passed")

## 6. 학습 곡선 시각화

`plot_training_log`는 `Trainer.fit`이 반환하는 epoch별 로그 리스트를 받아 loss와 metric 두 곡선을 PNG 파일로 저장한다.
`plt.show()` 없이 파일만 저장하므로 CLI 스크립트에서도 그래픽 환경 없이 동작한다.
반환값은 저장된 파일의 경로 문자열이다.

In [ ]:
# synthetic training log (20 epochs)
num_epochs = 20
np.random.seed(0)

logs = []
for epoch in range(1, num_epochs + 1):
    train_loss = 1.0 / epoch + np.random.rand() * 0.05
    test_loss  = 1.0 / epoch + np.random.rand() * 0.08 + 0.02
    train_metric = 1 - 0.5 / epoch + np.random.rand() * 0.02
    test_metric  = 1 - 0.5 / epoch - np.random.rand() * 0.02
    logs.append({
        "epoch": epoch,
        "train": {"loss": train_loss, "metric": train_metric},
        "test":  {"loss": test_loss,  "metric": test_metric},
    })

print(f"log count: {len(logs)}")
print(f"epoch 1:  train loss={logs[0]['train']['loss']:.4f}, test loss={logs[0]['test']['loss']:.4f}")
print(f"epoch 20: train loss={logs[-1]['train']['loss']:.4f}, test loss={logs[-1]['test']['loss']:.4f}")

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    saved_path = plot_training_log(
        logs,
        output_dir=tmpdir,
        filename="training_log.png"
    )
    print(f"saved: {saved_path}")

    # display inline for notebook
    from IPython.display import Image
    display(Image(saved_path))

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = plot_training_log(logs, output_dir=tmpdir, filename="check.png")

    assert isinstance(path, str), "return type must be str"
    assert pathlib.Path(path).exists(), "png file not created"
    assert path.endswith("check.png"), "filename not reflected in path"

print("plot_training_log validation passed")

## 요약

이 노트북에서 확인한 내용은 다음과 같다.

| 모듈 | API | 핵심 확인 사항 |
|---|---|---|
| `batching.py` | `get_batches` | 배열 1개는 ndarray, 2개 이상은 tuple yield; shuffle 시 인덱스 동기화 |
| `random.py` | `set_seed` | 동일 시드 재현, 다른 시드 비재현 |
| `io.py` | `save_params`, `load_params` | .npz 저장·복원, 키·값 일치 |
| `checkpoints.py` | `save`, `load` | in-place 복원, .npz 자동 확장자 처리 |
| `training_plots.py` | `plot_training_log` | PNG 저장, 반환 경로 정확성 |

**핵심 설계 원칙**
- `get_batches`는 복수 배열에 동일 인덱스를 적용하여 이미지-레이블 쌍 관계를 보장한다.
- `checkpoints.load`는 `param[...] =` in-place 방식으로 배열 참조를 교체하지 않고 메모리를 덮어쓴다.
- CuPy 배열은 `checkpoints.save`에서 자동으로 NumPy로 변환되어 `.npz`에 저장된다.